In [1]:
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 54.9 MB/s eta 0:00:00


In [2]:
import os
import pickle
import faiss
import numpy as np
import pandas as pd

from tqdm import tqdm

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
DATA="/content/drive/MyDrive/FIFI_Research/data"
MODEL_PATH="/content/drive/MyDrive/FIFI_Research/models"

train_df=pd.read_csv(
    os.path.join(DATA,"train.tsv"),
    sep="\t"
)

val_df=pd.read_csv(
    os.path.join(DATA,"val.tsv"),
    sep="\t"
)

print(train_df.shape)
print(val_df.shape)

(90000, 4)
(18000, 4)


In [4]:
with open(
    os.path.join(MODEL_PATH,"candidate_titles.pkl"),
    "rb"
) as f:
    candidate_titles=pickle.load(f)

candidate_embeddings=np.load(
    os.path.join(MODEL_PATH,"candidate_embeddings.npy")
)

index=faiss.read_index(
    os.path.join(MODEL_PATH,"faiss.index")
)

print(len(candidate_titles))
print(candidate_embeddings.shape)

2000
(2000, 768)


In [5]:
bge=SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
query_embeddings=bge.encode(

    val_df["generated_title"].tolist(),

    normalize_embeddings=True,

    batch_size=64,

    show_progress_bar=True
)

Batches:   0%|          | 0/282 [00:00<?, ?it/s]

In [7]:
scores,indices=index.search(

    query_embeddings,

    100
)

In [8]:
reranker=CrossEncoder(

    "cross-encoder/ms-marco-MiniLM-L-6-v2",

    max_length=256
)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [9]:
TOP_K = 20

reranked_titles = []

for i in tqdm(range(len(val_df))):

    query = val_df.iloc[i]["generated_title"]

    candidates = [
        candidate_titles[idx]
        for idx in indices[i][:TOP_K]
    ]

    pairs = [
        [query, title]
        for title in candidates
    ]

    scores = reranker.predict(
        pairs,
        batch_size=32
    )

    order = np.argsort(scores)[::-1]

    reranked = [
        candidates[j]
        for j in order
    ]

    reranked_titles.append(reranked)

100%|██████████| 18000/18000 [1:21:14<00:00,  3.69it/s]


In [10]:
def reciprocal_rank(predictions, gt):

    for rank, title in enumerate(predictions[:10], 1):

        if title == gt:

            return 1 / rank

    return 0


rr = []

for i in range(len(val_df)):

    rr.append(
        reciprocal_rank(
            reranked_titles[i],
            val_df.iloc[i]["original_title"]
        )
    )

print("MRR@10 =", np.mean(rr))

MRR@10 = 0.7699600529100529


In [11]:
results = []

for style in ["technical", "accessible", "catchy"]:

    subset = val_df[val_df["category"] == style]

    scores = []

    for idx in subset.index:

        scores.append(
            reciprocal_rank(
                reranked_titles[idx],
                val_df.loc[idx, "original_title"]
            )
        )

    results.append({
        "Style": style,
        "MRR@10": np.mean(scores)
    })

pd.DataFrame(results)

,Style,MRR@10
0,technical,0.914283
1,accessible,0.638780
2,catchy,0.756817


In [12]:
submission = []

for i, row in val_df.iterrows():

    item = {
        "id": row["id"]
    }

    for k in range(10):

        item[f"rank{k+1}"] = reranked_titles[i][k]

    submission.append(item)

submission = pd.DataFrame(submission)

submission.to_csv(
    "/content/drive/MyDrive/FIFI_Research/submissions/FutureMinds_task1_run3.tsv",
    sep="\t",
    index=False
)

print(submission.head())

   id                                              rank1  \
0   0  Preterm infants' limb-pose estimation from dep...   
1   1  DSGAN: Generative Adversarial Training for Dis...   
2   2  Folding-based compression of point cloud attri...   
3   3      Copy that! Editing Sequences by Copying Spans   
4   4  ReLLIE: Deep Reinforcement Learning for Custom...   

                                               rank2  \
0  Back to the Future: Joint Aware Temporal Deep ...   
1  Denoising Relation Extraction from Document-le...   
2  DeepCompress: Efficient Point Cloud Geometry C...   
3  Unadversarial Examples: Designing Objects for ...   
4  Depth Image Upsampling based on Guided Filter ...   

                                               rank3  \
0  Peeking into occluded joints: A novel framewor...   
1  Learning Open Information Extraction of Implic...   
2  Rethinking Sampling in 3D Point Cloud Generati...   
3  CAD Priors for Accurate and Flexible Instance ...   
4                 Supe

In [14]:
print("reranked_titles" in globals())
print("reranked_scores" in globals())

True
False


In [15]:
submission = []

for i, row in val_df.iterrows():

    item = {
        "id": row["id"]
    }

    for k in range(10):

        item[f"rank{k+1}"] = reranked_titles[i][k]

    submission.append(item)

submission = pd.DataFrame(submission)

print(submission.head())

print(submission.shape)

submission.to_csv(
    "/content/drive/MyDrive/FIFI_Research/submissions/FutureMinds_task1_run3.tsv",
    sep="\t",
    index=False
)

print("✅ Task1 Run3 Saved Successfully")

   id                                              rank1  \
0   0  Preterm infants' limb-pose estimation from dep...   
1   1  DSGAN: Generative Adversarial Training for Dis...   
2   2  Folding-based compression of point cloud attri...   
3   3      Copy that! Editing Sequences by Copying Spans   
4   4  ReLLIE: Deep Reinforcement Learning for Custom...   

                                               rank2  \
0  Back to the Future: Joint Aware Temporal Deep ...   
1  Denoising Relation Extraction from Document-le...   
2  DeepCompress: Efficient Point Cloud Geometry C...   
3  Unadversarial Examples: Designing Objects for ...   
4  Depth Image Upsampling based on Guided Filter ...   

                                               rank3  \
0  Peeking into occluded joints: A novel framewor...   
1  Learning Open Information Extraction of Implic...   
2  Rethinking Sampling in 3D Point Cloud Generati...   
3  CAD Priors for Accurate and Flexible Instance ...   
4                 Supe

In [16]:
submission = []

for i, row in val_df.iterrows():

    query_id = row["id"]

    for rank in range(10):

        submission.append({

            "id": query_id,

            "rank": rank + 1,

            "score": round(1.0 - rank * 0.001, 3),

            "original_title": reranked_titles[i][rank]

        })

submission = pd.DataFrame(submission)

print(submission.head(20))

print(submission.shape)

submission.to_csv(

    "/content/drive/MyDrive/FIFI_Research/submissions/FutureMinds_task1_run3.tsv",

    sep="\t",

    index=False

)

print("✅ Task1 Run3 Saved Successfully")

    id  rank  score                                     original_title
0    0     1  1.000  Preterm infants' limb-pose estimation from dep...
1    0     2  0.999  Back to the Future: Joint Aware Temporal Deep ...
2    0     3  0.998  Peeking into occluded joints: A novel framewor...
3    0     4  0.997  Real-time Deep Pose Estimation with Geodesic L...
4    0     5  0.996  Depth Estimation from Single Image using Spars...
5    0     6  0.995  Capture Dense: Markerless Motion Capture Meets...
6    0     7  0.994  Collaborative Descriptors: Convolutional Maps ...
7    0     8  0.993  Iterative Multi-domain Regularized Deep Learni...
8    0     9  0.992  BWCNN: Blink to Word, a Real-Time Convolutiona...
9    0    10  0.991  A Deep Framework for Bone Age Assessment based...
10   1     1  1.000  DSGAN: Generative Adversarial Training for Dis...
11   1     2  0.999  Denoising Relation Extraction from Document-le...
12   1     3  0.998  Learning Open Information Extraction of Implic...
13   1